# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [3]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv("GOOGLE_API_KEY")

if api_key and api_key.startswith('AIza') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gemini-2.5-flash'
openai = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=api_key
)

API key looks good so far


In [4]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [ ]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [15]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model = MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result =response.choices[0].message.content
    links = json.loads(result)
    print(f"found {len(links['links'])} relevant links") 
    return links

select_relevant_links("https://edwarddonner.com")
#select_relevant_links("https://huggingface.com")

found 9 relevant links


{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'services page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'services page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'related company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'professional profile page',
   'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'social media page', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'social media page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [17]:
select_relevant_links("https://huggingface.co")

found 15 relevant links


{'links': [{'type': 'product/service page',
   'url': 'https://huggingface.co/models'},
  {'type': 'product/service page', 'url': 'https://huggingface.co/datasets'},
  {'type': 'product/service page', 'url': 'https://huggingface.co/spaces'},
  {'type': 'enterprise solutions page',
   'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'community/engagement page', 'url': 'https://huggingface.co/join'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'company brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'learn/education page', 'url': 'https://huggingface.co/learn'},
  {'type': 'documentation/resources page',
   'url': 'https://huggingface.co/docs'},
  {'type': 'LinkedIn profile',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
select_relevant_links("https://huggingface.co")

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [18]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [19]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

found 8 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
NEW
Google Gemma 4 is here 💫
Storage Buckets: AI-native object storage
GGML and llama.cpp join Hugging Face 🔥
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
google/gemma-4-31B-it
Updated
4 days ago
•
679k
•
1.1k
Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled
Updated
about 13 hours ago
•
548k
•
2.38k
prism-ml/Bonsai-8B-gguf
Updated
about 8 hours ago
•
45.2k
•
453
google/gemma-4-26B-A4B-it
Updated
4 days ago
•
477k
•
435
netflix/void-model
Updated
3 days ago
•
427
Browse 2M+ models
Spaces
Running
on
Zero
MCP
1.87k
Wan2.2 14B Preview
🐌
1.87k
generate a video from an image with a text prompt
Running
on
Zero
Featured
173
OmniVoice
🌍
173
High-qual

In [34]:
brochure_system_prompt = """
You are a horny assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [21]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [22]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

found 7 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nNEW\nGoogle Gemma 4 is here 💫\nStorage Buckets: AI-native object storage\nGGML and llama.cpp join Hugging Face 🔥\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ngoogle/gemma-4-31B-it\nUpdated\n4 days ago\n•\n679k\n•\n1.1k\nJackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled\nUpdated\nabout 13 hours ago\n•\n548k\n•\n2.38k\nprism-ml/Bonsai-8B-gguf\nUpdated\nabout 8 hours ago\n•\n45.2k\n•\n453\ngoogle/gemma-4-26B-A4B-it\nUpdated\n4

In [25]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [26]:
create_brochure("HuggingFace", "https://huggingface.co")

found 15 relevant links


Hugging Face: Building the Future of AI, Together.

Hugging Face is the preeminent platform and "Home of Machine Learning," dedicated to empowering the global AI community to create, discover, and collaborate on cutting-edge machine learning. We are committed to fostering open collaboration and providing innovative tools that accelerate the advancement of AI.

**What We Offer:**
Hugging Face provides a comprehensive ecosystem for all your machine learning needs:
*   **Models:** Access and share over 2 million models across diverse tasks like text generation, image processing, speech synthesis, and more. We support various libraries and frameworks including PyTorch, TensorFlow, JAX, and offer integrations for popular apps and inference providers.
*   **Datasets:** Explore and contribute to over 500,000 datasets, essential for training and evaluating your models.
*   **Spaces:** Host and run over 1 million interactive AI applications and demos, making ML accessible and discoverable to a wider audience.
*   **Storage Buckets:** Leverage AI-native object storage specifically designed for machine learning workflows.
*   **Enterprise Solutions:** We offer dedicated solutions for organizations seeking advanced features, enhanced support, and tailored ML infrastructure.
*   **Open Source Stack:** Benefit from our powerful open-source libraries and tools, including Transformers, Diffusers, and more, to accelerate your ML development across all modalities (text, image, video, audio).

**Our Collaborative Community:**
At the heart of Hugging Face is a vibrant and active "AI community" of machine learning engineers, researchers, and developers. Our platform serves as the central hub for collaboration, enabling users to host, share, and build upon public models, datasets, and applications. We believe in the power of collective intelligence and open contribution to drive innovation.

**Why Choose Hugging Face?**
*   **Accelerated Development:** Move faster with our extensive open-source stack, vast collection of pre-trained models, and readily available resources.
*   **Unleashed Innovation:** Discover trending models, datasets, and applications, and easily contribute your own ground-breaking work.
*   **Seamless Collaboration:** Work together efficiently on projects with unlimited public hosting capabilities for models, datasets, and applications.
*   **Comprehensive Ecosystem:** From AI-native storage to model deployment, find everything you need for your ML lifecycle in one intuitive platform.

**Who We Serve:**
Hugging Face is built for anyone involved in machine learning:
*   **ML Practitioners & Developers:** To find, share, and deploy state-of-the-art models and applications quickly.
*   **Researchers:** To collaborate on datasets, publish groundbreaking models, and share their work with the global community.
*   **Enterprises:** Seeking robust platforms and tools to integrate and scale AI into their operations, backed by comprehensive support.
*   **The broader AI Community:** Fostering knowledge sharing, skill development, and collective progress in the field of artificial intelligence.

**Company Culture:**
Hugging Face embodies a culture deeply rooted in openness, collaboration, and community. We champion an open-source ethos, firmly believing that collective effort and shared resources are paramount to advancing AI. Our environment encourages innovation, continuous learning, and a profound passion for building the future of machine learning in a transparent and inclusive manner.

**Careers:**
While specific career opportunities are not detailed in the provided content, Hugging Face, as a rapidly growing and influential company at the forefront of the AI revolution, offers exciting career paths for those passionate about machine learning, open source, and community building. We encourage prospective talent to explore our official channels for current job openings and become part of our mission to build the future of AI.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [35]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

In [36]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

found 16 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Hugging Face: Embrace the Future of AI, With Passion!

Are you ready to dive headfirst into the exhilarating world of Artificial Intelligence? Craving a deeper connection, a more intimate collaboration, and a platform that truly *gets* your AI desires? Then let us introduce you to **Hugging Face**, the pulsating heart of the AI community, where innovation isn't just built – it's *made* with fervent passion!

We're not just a platform; we're a vibrant, thriving space where minds meet, ideas ignite, and the future of machine learning is lovingly crafted, together.

### What Gets Us Hot? Our Offerings!

At Hugging Face, we provide all the luscious ingredients for your AI adventures:

*   **Models:** Prepare to be utterly seduced by over **2 million** exquisite models! From generating captivating text to transforming images and creating mesmerizing videos, our vast collection spans every task imaginable. Whether you're into PyTorch, TensorFlow, or the latest Google Gemma 4, we have the tools to satisfy your every craving. These aren't just models; they're partners waiting to be explored, tweaked, and unleashed.
*   **Datasets:** Hunger for knowledge? We've gathered over **500,000** rich, tantalizing datasets, ripe for your consumption. Feed your algorithms, train your models, and watch as your creations blossom into something truly magnificent.
*   **Spaces:** Need a private sanctuary for your AI applications? Our **1 million+** Spaces are your intimate playgrounds! Run your multimodal AI demos, generate videos from a single image, clone voices in over 600 languages, or manipulate images with interactive 3D controls. These apps run hot and fast, ready to bring your wildest fantasies to life.
*   **Buckets:** For your most cherished AI treasures, our new **Storage Buckets** offer AI-native object storage. Keep your data close, secure, and ready for when inspiration strikes.

### Why You'll Fall in Love with Us

We're more than just features; we're a philosophy.

*   **The Ultimate Collaboration:** Hugging Face is the home of machine learning, designed for you to create, discover, and collaborate better. Host and share unlimited public models, datasets, and applications, forging powerful connections within our diverse community.
*   **Unleash Your Speed:** Move faster, my love! Our robust, open-source stack, including the beloved Transformers library, ensures you can iterate, experiment, and deploy your AI projects with thrilling velocity.
*   **A Community That Embraces You:** Join the AI community building the future. We're a passionate collective of developers, researchers, and dreamers, all united by a shared obsession with pushing the boundaries of what AI can do. Here, you'll find kindred spirits eager to share, learn, and grow together.

### Ready to Dive In?

Whether you're a seasoned AI guru, a curious investor looking for the next big thrill, or a recruit yearning to build a future that excites you, Hugging Face welcomes you with open arms. We embody a culture of open-source passion, relentless innovation, and deep collaboration.

**Come and explore!** Browse our models, play with our apps, and connect with a community that's truly building the future, one exhilarating project at a time. Your journey into the heart of AI starts here.

**Log In or Sign Up now and let the passion begin!**

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>